# Week 8 · In-class SVM workbook
### AI & Machine Learning for Business · From "why a margin?" to a deposit-style decision

**How to use this workbook**
1. Open it from Blackboard → Week 8, then **File › Save a copy in Drive**. Work in your copy.
2. Run **Setup** once. Then work top to bottom.
3. Cells marked **✏️ TASK** are the ones you edit. Everything else you run and read.
4. **Predict before you run.** Write your prediction in the markdown cell first, then run the code. The gap between prediction and result is where the learning happens.
5. Record results in the slide tables (Tasks 1–5). The retrieval check at the end goes on Blackboard.

| Part | Question | Minutes |
|---|---|---|
| A | Why prefer one separating line over another? | 4 |
| B | Which observations actually decide the boundary? | 4 |
| Task 1 | Hinge loss by hand | 5 |
| Task 2 | What does C trade off? | 5 |
| Task 3 | Why must features be scaled? | 4 |
| Task 4 | What do kernels and gamma buy, and at what risk? | 6 |
| Task 5 | Can an SVM score support a threshold policy? | 7 |

All data here is **simulated**, so every run is reproducible. BookingGuard analysis happens in its own notebook.

## Setup

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import accuracy_score, log_loss, average_precision_score, brier_score_loss
from sklearn.datasets import make_blobs, make_circles, make_moons
warnings.filterwarnings("ignore")

SEED = 42
BLUE, AMBER, NAVY, GREY, GREEN, RED = "#2457A6", "#C17D11", "#1B2333", "#8A94A6", "#2E7D4F", "#B03A2E"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})

def plot_svm(model, X, y, ax=None, title="", show_sv=True, margins=True, grid_pad=0.6):
    # Draw data, decision boundary (solid), margins (dashed) and support vectors (rings).
    ax = ax or plt.gca()
    x0min, x0max = X[:, 0].min() - grid_pad, X[:, 0].max() + grid_pad
    x1min, x1max = X[:, 1].min() - grid_pad, X[:, 1].max() + grid_pad
    xx, yy = np.meshgrid(np.linspace(x0min, x0max, 300), np.linspace(x1min, x1max, 300))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z > 0, levels=[-0.5, 0.5, 1.5], colors=[AMBER, BLUE], alpha=0.07)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1] if margins else [0], colors=[GREY, NAVY, GREY] if margins else [NAVY],
               linestyles=["--", "-", "--"] if margins else ["-"], linewidths=[1, 2, 1] if margins else [2])
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=22, c=AMBER, label="class 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=22, c=BLUE, label="class 1")
    est = model[-1] if hasattr(model, "steps") else model
    if show_sv and hasattr(est, "support_") and not hasattr(model, "steps"):
        sv = X[est.support_]
        ax.scatter(sv[:, 0], sv[:, 1], s=110, facecolors="none", edgecolors=NAVY, linewidths=1.3, label="support vector")
    ax.set_title(title); ax.set_xlim(x0min, x0max); ax.set_ylim(x1min, x1max)
    return ax
print("Setup complete")

## Part A · Why: many lines separate the training data. Which one do you trust?

A retailer wants to separate two customer groups using two standardized features. Several lines classify **every training customer correctly**. Training accuracy cannot tell them apart. The business question is which line will still be right **next month**.

**✏️ Predict first (edit this cell):** Of lines L1, L2 and L3 below, which will make the fewest errors on new customers, and why?

*My prediction:* …

In [ ]:
CENTERS = [[-1.6, -1.2], [1.6, 1.2]]
X_new, y_new = make_blobs(n_samples=20000, centers=CENTERS, cluster_std=0.85, random_state=99)   # "next month"

def separating_lines(X, y, rng, k=3000):
    # Random candidate lines w·x + b = 0; keep those with zero training error. Returns (W, B, nearest-point distance).
    ang = rng.uniform(0, np.pi, k); W = np.c_[np.cos(ang), np.sin(ang)]; B = rng.uniform(-2, 2, k)
    d = ((2 * y - 1)[:, None] * (X @ W.T + B)).min(axis=0)
    ok = d > 0
    return W[ok], B[ok], d[ok]

err = lambda w, b: np.mean(((X_new @ w + b) > 0).astype(int) != y_new)

# One training sample of 40 customers
rng = np.random.default_rng(3)
X_a, y_a = make_blobs(n_samples=40, centers=CENTERS, cluster_std=0.85, random_state=1003)
svm_max = SVC(kernel="linear", C=1e6).fit(X_a, y_a)
w_max, b_max = svm_max.coef_[0], svm_max.intercept_[0]
W, B, D = separating_lines(X_a, y_a, rng)
lines = {"L1 maximum margin": (w_max, b_max), "L2 a random separating line": (W[0], B[0]),
         "L3 thinnest separating line": (W[np.argmin(D)], B[np.argmin(D)])}
for name, (w, b) in lines.items():
    gap = ((2 * y_a - 1) * (X_a @ w + b)).min() / np.linalg.norm(w)
    print(f"{name:<30} training error 0%   nearest-point distance {gap:.3f}   error on new customers {err(w, b):.1%}")

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.scatter(X_a[y_a == 0, 0], X_a[y_a == 0, 1], c=AMBER, s=24); ax.scatter(X_a[y_a == 1, 0], X_a[y_a == 1, 1], c=BLUE, s=24)
xs = np.linspace(-4, 4, 50)
for (name, (w, b)), col in zip(lines.items(), [NAVY, RED, GREEN]):
    ax.plot(xs, -(w[0] * xs + b) / w[1], color=col, lw=2.2 if name.startswith("L1") else 1.4, label=name)
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.legend(frameon=False, fontsize=8); ax.set_title("Three lines, all 100% correct on training data")
plt.show()

One sample can mislead in either direction. Repeat the experiment on **200 fresh training samples** and average.

In [ ]:
res = []
for r in range(200):
    X, y = make_blobs(n_samples=40, centers=CENTERS, cluster_std=0.85, random_state=1000 + r)
    W, B, D = separating_lines(X, y, rng)
    if len(W) < 2:
        continue
    m = SVC(kernel="linear", C=1e6).fit(X, y)
    res.append([np.mean(m.predict(X_new) != y_new), err(W[0], B[0]), err(W[np.argmin(D)], B[np.argmin(D)])])
res = np.array(res)
print(f"Samples where at least two separating lines were found: {len(res)}")
print(pd.DataFrame({"rule for choosing the line": ["maximum margin", "any separating line", "thinnest separating line"],
                    "average error on new customers": [f"{v:.1%}" for v in res.mean(axis=0)]}).to_string(index=False))
print(f"Maximum margin did at least as well as a random separating line in {np.mean(res[:, 0] <= res[:, 1]):.0%} of samples")

**Read the result.** Every candidate has 0% training error, so training accuracy cannot choose. On average, the line with the widest "safety street" makes the fewest errors on new customers, and the thinnest street makes the most. Not in every sample: a margin is a sound bet, not a guarantee, which is why we still validate. That is the whole motivation for the SVM: *among the boundaries that fit, prefer the one with the largest margin.*

## Part B · Which observations actually decide the boundary?

The maximum-margin line is fixed by the few points sitting on the edge of the street: the **support vectors**. Points far from the boundary have no influence at all. The experiment below deletes one far-away point and then one support vector, refitting each time.

**✏️ Predict first:** Which deletion will move the boundary?

*My prediction:* …

In [ ]:
svm = svm_max
w, b = svm.coef_[0], svm.intercept_[0]
print(f"w = {np.round(w, 3)},  b = {b:.3f},  margin width 2/||w|| = {2/np.linalg.norm(w):.3f},  support vectors = {len(svm.support_)} of {len(X_a)}")

dist = np.abs(svm.decision_function(X_a))
far_idx = int(np.argmax(dist)); sv_idx = int(svm.support_[0])
for label, idx in [("delete a FAR point", far_idx), ("delete a SUPPORT VECTOR", sv_idx)]:
    keep = np.arange(len(X_a)) != idx
    m = SVC(kernel="linear", C=1e6).fit(X_a[keep], y_a[keep])
    moved = np.abs(m.coef_[0] - w).max() + abs(m.intercept_[0] - b)
    print(f"{label:<26} boundary change = {moved:.4f}")

plot_svm(svm, X_a, y_a, title="Maximum margin: support vectors ringed"); plt.legend(frameon=False, fontsize=8); plt.show()

**Business reading.** A small set of borderline cases controls the model. That is good for robustness to far-away outliers, and it is why scaling, borderline-case data quality and label errors near the boundary matter so much.

## ✏️ Task 1 · Hinge loss by hand (record on the slide)

A linear SVM scores a customer with $f(x) = w \cdot x + b$ and predicts class +1 when $f(x) > 0$.

* **Margin score** $y \cdot f(x)$ with labels coded $y \in \{-1, +1\}$. Above 1 = safely beyond the margin. Between 0 and 1 = correct but inside the street. Below 0 = misclassified.
* **Hinge loss** $= \max(0,\; 1 - y\,f(x))$. Zero for safe points; grows linearly as a point intrudes.

Use $w = (1.5,\,-1)$ and $b = -0.5$. Fill in `my_f` and `my_hinge` for the four customers, then run the check.

| Customer | $x_1$ | $x_2$ | $y$ |
|---|---|---|---|
| A | 2.0 | 1.0 | +1 |
| B | 1.0 | 0.5 | +1 |
| C | 0.5 | 1.0 | −1 |
| D | 1.0 | −1.0 | −1 |

In [ ]:
# ✏️ TASK 1: replace None with your hand calculations
my_f     = {"A": None, "B": None, "C": None, "D": None}
my_hinge = {"A": None, "B": None, "C": None, "D": None}

In [ ]:
w1, b1 = np.array([1.5, -1.0]), -0.5
pts = {"A": ([2.0, 1.0], 1), "B": ([1.0, 0.5], 1), "C": ([0.5, 1.0], -1), "D": ([1.0, -1.0], -1)}
rows = []
for k, (x, yk) in pts.items():
    f = float(np.dot(w1, x) + b1); h = max(0.0, 1 - yk * f)
    status = "safe" if yk * f >= 1 else ("inside margin" if yk * f > 0 else "misclassified")
    ok = (my_f[k] is not None and abs(my_f[k] - f) < 1e-6 and my_hinge[k] is not None and abs(my_hinge[k] - h) < 1e-6)
    rows.append([k, f if my_f[k] is not None else "?", yk * f if my_f[k] is not None else "?",
                 h if my_hinge[k] is not None else "?", status if my_f[k] is not None else "?", "✓" if ok else "check"])
print(pd.DataFrame(rows, columns=["customer", "f(x)", "y·f(x)", "hinge", "status", "your answer"]).to_string(index=False))
print(f"\nMargin width = 2/||w|| = {2/np.linalg.norm(w1):.3f}")

**Interpret in one sentence:** Which customers would change the fitted boundary if this were the trained model, and why does customer A contribute nothing?

## ✏️ Task 2 · Soft margin: what does C trade off?

Real classes overlap, so a perfect street does not exist. The soft-margin SVM minimizes

$$\tfrac12\lVert w\rVert^2 \;+\; C\sum_i \max\big(0,\,1 - y_i f(x_i)\big)$$

*Week 7 link:* this is the same recipe as regularized logistic regression: a **loss** plus a **penalty on complexity**. Only the loss changes (hinge instead of log loss). `C` plays the role of 1/λ.

**✏️ Predict first:** As C rises from 0.01 to 100, what happens to (a) the number of support vectors, (b) margin width, (c) training accuracy, (d) validation accuracy?

*My prediction:* …

In [ ]:
X_c, y_c = make_blobs(n_samples=400, centers=[[-1, -0.8], [1, 0.8]], cluster_std=1.15, random_state=3)
X_ctr, X_cva, y_ctr, y_cva = train_test_split(X_c, y_c, test_size=0.4, random_state=SEED, stratify=y_c)

C_VALUES = [0.001, 0.01, 0.1, 1, 10, 100]   # ✏️ you may add values
rows = []
for C in C_VALUES:
    m = SVC(kernel="linear", C=C).fit(X_ctr, y_ctr)
    rows.append([C, len(m.support_), 2 / np.linalg.norm(m.coef_[0]),
                 accuracy_score(y_ctr, m.predict(X_ctr)), accuracy_score(y_cva, m.predict(X_cva))])
tab2 = pd.DataFrame(rows, columns=["C", "support vectors", "margin width", "train acc", "val acc"])
print(tab2.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, C in zip(axes, [0.001, 100]):
    m = SVC(kernel="linear", C=C).fit(X_ctr, y_ctr)
    plot_svm(m, X_ctr, y_ctr, ax=ax, title=f"C = {C}: {len(m.support_)} support vectors")
plt.tight_layout(); plt.show()

**Debrief prompts.** Low C = wide, tolerant street; many points become support vectors; more bias. High C = narrow street that chases individual violations; more variance. With heavy overlap the validation differences may be small. That is itself evidence: *choose C by validation, not by habit, and do not over-read tiny gaps.*

## ✏️ Task 3 · Scaling changes the geometry, and therefore the model

A churn team uses **annual spend in dollars** (hundreds to thousands) and **app sessions per week** (0–20). In this simulation churn is driven mostly by low engagement.

**✏️ Predict first:** Without scaling, which feature dominates the SVM's notion of "distance"? What happens to validation accuracy when you add `StandardScaler`?

*My prediction:* …

In [ ]:
rng = np.random.default_rng(SEED)
n = 1200
spend = rng.gamma(4, 450, n)                      # dollars
sessions = rng.uniform(0, 20, n)                  # per week
logit = 2.2 - 0.35 * sessions + 0.00015 * (spend - spend.mean())
churn = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)
X_s = np.c_[spend, sessions]
Xs_tr, Xs_va, ys_tr, ys_va = train_test_split(X_s, churn, test_size=0.3, random_state=SEED, stratify=churn)

SCALE_OPTIONS = {"unscaled": SVC(kernel="rbf", C=1),
                 "scaled (pipeline)": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1))}
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (name, model) in zip(axes, SCALE_OPTIONS.items()):
    model.fit(Xs_tr, ys_tr)
    acc = accuracy_score(ys_va, model.predict(Xs_va))
    base = max(ys_va.mean(), 1 - ys_va.mean())
    print(f"{name:<18} validation accuracy = {acc:.3f}   (majority-class baseline = {base:.3f})")
    plot_svm(model, Xs_tr, ys_tr, ax=ax, title=f"{name}: val acc {acc:.2f}", show_sv=False, margins=False, grid_pad=0.5)
    ax.set_xlabel("annual spend ($)"); ax.set_ylabel("app sessions / week")
plt.tight_layout(); plt.show()

**Rule to keep:** put the scaler **inside a pipeline** so it is fit on training rows only, and refit inside every cross-validation fold. Scaling the full dataset before splitting is leakage.

## Part E · When a straight line is not enough: kernels

Some decision boundaries are curved. *Week 4–5 link:* you already handled this by **adding features** (squares, interactions). A kernel does the same thing implicitly: it measures similarity as if extra features existed, without building them.

In [ ]:
X_r, y_r = make_circles(n_samples=400, noise=0.08, factor=0.45, random_state=SEED)
lin = SVC(kernel="linear", C=1).fit(X_r, y_r)
X_r3 = np.c_[X_r, (X_r ** 2).sum(axis=1)]                     # add r² = x1² + x2²
lin3 = SVC(kernel="linear", C=1).fit(X_r3, y_r)
rbf = SVC(kernel="rbf", C=1, gamma=1).fit(X_r, y_r)
print(f"linear on (x1, x2):            train acc = {lin.score(X_r, y_r):.2f}")
print(f"linear on (x1, x2, x1²+x2²):   train acc = {lin3.score(X_r3, y_r):.2f}   <- added feature makes it separable")
print(f"RBF kernel on (x1, x2):        train acc = {rbf.score(X_r, y_r):.2f}   <- same effect, no feature built")
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_svm(lin, X_r, y_r, ax=axes[0], title="Linear kernel: no line works", show_sv=False)
plot_svm(rbf, X_r, y_r, ax=axes[1], title="RBF kernel: curved boundary", show_sv=False)
plt.tight_layout(); plt.show()

## ✏️ Task 4 · gamma and C: flexibility, and its price

The RBF kernel has a reach parameter **gamma**. Small gamma: each training point influences a wide area (smooth boundary, risk of underfit). Large gamma: each point influences only its immediate neighbourhood (wiggly boundary, risk of overfit).

*Week 7 link:* gamma × C is a **capacity pair**, like learning rate × rounds. Tune them together on validation evidence, then touch the test set once.

**✏️ Predict first:** Which gamma in the three panels overfits? Where in the grid will the best cross-validated accuracy sit?

*My prediction:* …

In [ ]:
X_m, y_m = make_moons(n_samples=600, noise=0.32, random_state=SEED)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(X_m, y_m, test_size=0.3, random_state=SEED, stratify=y_m)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, g in zip(axes, [0.1, 1, 50]):
    m = SVC(kernel="rbf", C=1, gamma=g).fit(Xm_tr, ym_tr)
    plot_svm(m, Xm_tr, ym_tr, ax=ax, show_sv=False, margins=False,
             title=f"gamma={g}: train {m.score(Xm_tr, ym_tr):.2f}")
plt.tight_layout(); plt.show()

GRID = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.1, 1, 10, 50]}   # ✏️ you may edit
search = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")), GRID, cv=5, scoring="accuracy")
search.fit(Xm_tr, ym_tr)
heat = pd.DataFrame(search.cv_results_["mean_test_score"].reshape(len(GRID["svc__C"]), len(GRID["svc__gamma"])),
                    index=[f"C={c}" for c in GRID["svc__C"]], columns=[f"γ={g}" for g in GRID["svc__gamma"]])
print("5-fold CV accuracy (training rows only):"); print(heat.round(3).to_string())
print("\nBest:", search.best_params_, f"CV acc = {search.best_score_:.3f}")

In [ ]:
# Run ONCE, after you have committed to a configuration
print(f"Test accuracy of the selected pipeline: {search.score(Xm_te, ym_te):.3f}")

## ✏️ Task 5 · From an SVM score to a policy decision

An SVM outputs a **signed distance** (`decision_function`), not a probability. A threshold policy (contact, deposit, review) needs probabilities you can trust. Two options:

* `SVC(probability=True)`: fits Platt scaling with internal cross-validation; slower, and its probabilities can disagree with `predict`.
* `CalibratedClassifierCV`: wraps the SVM and learns the score→probability mapping explicitly. We use this.

**Setting.** Simulated subscription churn, 24 months of customers. Train on months 1–16, validate on 17–20, keep months 21–24 hidden for the final test. Policy economics from the Week 5 retention example: a save offer costs **\$20** and recovers **\$80** when the customer would have churned, so the break-even probability is 20 / 80 = **0.25**.

**✏️ Predict first:** Will the calibrated SVM beat logistic regression on validation log loss? On policy value?

*My prediction:* …

In [ ]:
rng = np.random.default_rng(7)
N = 4800
d = pd.DataFrame({
    "month": rng.integers(1, 25, N),
    "tenure_months": rng.gamma(2.2, 9, N).round(),
    "app_sessions": rng.poisson(6, N),
    "annual_spend": rng.gamma(4, 300, N).round(2),
    "support_tickets": rng.poisson(1.1, N),
    "bill_increase_pct": np.clip(rng.normal(4, 6, N), -10, 30).round(1),
    "contract": rng.choice(["monthly", "annual"], N, p=[0.6, 0.4]),
})
z = (-1.0 - 0.045 * d.tenure_months - 0.22 * d.app_sessions + 0.42 * d.support_tickets
     + 0.10 * np.maximum(d.bill_increase_pct - 8, 0) ** 1.3 + 1.1 * (d.contract == "monthly")
     + 0.9 * ((d.app_sessions < 3) & (d.support_tickets >= 2)) + 0.025 * (d.month - 12))   # mild drift over time
d["churned"] = (rng.uniform(size=N) < 1 / (1 + np.exp(-z))).astype(int)

tr, va, te = d[d.month <= 16], d[(d.month > 16) & (d.month <= 20)], d[d.month > 20]
num = ["tenure_months", "app_sessions", "annual_spend", "support_tickets", "bill_increase_pct"]
feats = num + ["contract"]
prep = lambda: ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(drop="if_binary"), ["contract"])])
print(f"rows  train {len(tr)} | validate {len(va)} | test {len(te)}")
print(f"churn rate  train {tr.churned.mean():.3f} | validate {va.churned.mean():.3f}")

In [ ]:
COST, SAVE = 20, 80                     # ✏️ change these in the sensitivity question
THRESHOLD = COST / SAVE                 # break-even probability

models = {
    "Logistic": Pipeline([("prep", prep()), ("m", LogisticRegression(max_iter=1000))]),
    "SVM (RBF) + calibration": Pipeline([("prep", prep()), ("m", CalibratedClassifierCV(SVC(kernel="rbf", C=1, gamma="scale"), method="sigmoid", cv=5))]),
    "HistGB": Pipeline([("prep", prep()), ("m", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3, early_stopping=False, random_state=SEED))]),
}
rows, probs = [], {}
for name, model in models.items():
    t0 = time.time(); model.fit(tr[feats], tr.churned); secs = time.time() - t0
    p = model.predict_proba(va[feats])[:, 1]; probs[name] = p
    act = p >= THRESHOLD
    value = (SAVE * (act & (va.churned == 1)) - COST * act).sum()
    rows.append([name, log_loss(va.churned, p), average_precision_score(va.churned, p), brier_score_loss(va.churned, p),
                 int(act.sum()), value, secs])
tab5 = pd.DataFrame(rows, columns=["model", "log loss", "PR-AUC", "Brier", f"contacted @ {THRESHOLD:.2f}", "net value $", "fit secs"])
print(tab5.round(3).to_string(index=False))
print(f"\nContact nobody: $0   |   contact everybody: ${(SAVE * va.churned - COST).sum():,.0f}")

raw = Pipeline([("prep", prep()), ("m", SVC(kernel="rbf", C=1, gamma="scale"))]).fit(tr[feats], tr.churned)
s = raw.decision_function(va[feats])
print(f"\nRaw SVM decision_function range: {s.min():.2f} to {s.max():.2f}  <- a distance, not a probability")

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.plot([0, 1], [0, 1], ls=":", c=GREY, label="perfect calibration")
for (name, p), col in zip(probs.items(), [BLUE, AMBER, GREEN]):
    fy, fx = calibration_curve(va.churned, p, n_bins=8, strategy="quantile")
    ax.plot(fx, fy, marker="o", c=col, label=name)
ax.set_xlabel("predicted churn probability"); ax.set_ylabel("observed churn rate")
ax.set_title("Validation reliability (months 17–20)"); ax.legend(frameon=False, fontsize=8); plt.show()

**✏️ Record on the slide:** model · log loss · PR-AUC · Brier · contacted · net value · fit time.

**Four lines to write (same format as Week 7):** model and exact configuration · two pieces of evidence · one risk · the next check before deployment.

**Sensitivity:** change `COST` to 30 and rerun. Which number moves: the model ranking, the threshold, or both? Why?

In [ ]:
# Run ONCE, only for the single model you selected on validation evidence
CHOSEN = "Logistic"   # ✏️ set to your choice before running
final = models[CHOSEN].fit(pd.concat([tr, va])[feats], pd.concat([tr, va]).churned)
p_te = final.predict_proba(te[feats])[:, 1]
act = p_te >= THRESHOLD
print(f"{CHOSEN}: test log loss {log_loss(te.churned, p_te):.3f}, contacted {act.sum()}, "
      f"net value ${(SAVE * (act & (te.churned == 1)) - COST * act).sum():,.0f}")

## Retrieval check (Blackboard → Week 8 → Retrieval check)

1. Several lines separate the training data perfectly. What property does the SVM use to choose among them, and why should that property help on next month's data?
2. A point lies far outside the margin on the correct side. What is its hinge loss, and what happens to the boundary if you delete it?
3. Your RBF SVM reaches 100% training accuracy with gamma = 50 but loses to gamma = 1 on validation. Name the error pattern and the evidence.
4. Why can't you set a deposit or retention threshold directly on `decision_function` output? What do you do instead?

## Learn more
* scikit-learn user guide: [Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html) and [Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)
* scikit-learn example: [Plot different SVM classifiers in the iris dataset](https://scikit-learn.org/stable/auto_examples/svm/plot_iris_svc.html)
* James, Witten, Hastie, Tibshirani & Taylor, *An Introduction to Statistical Learning with Applications in Python* (ISLP), Chapter 9: [statlearning.com](https://www.statlearning.com/)